# Tüm sonuçlar — dört grubu birleştir

Bu notebook, `1_baseline_cnn`, `2_baseline_vit`, `3_optimized_cnn` ve `4_optimized_vit` notebook'larının Drive'a kaydettiği sonuçları okur ve `ALL_RESULTS.xlsx` dosyasını üretir. GPU gerekmez.

Excel dosyasındaki sayfalar:
- **Test / Validation / Train (ort±std):** Her model için 5 fold'un ortalaması ± standart sapması. Makaledeki tablolar buradan alınır.
- **Veri bölünmesi:** Her fold'daki train, validation ve test sayıları (sınıf bazında).
- **Tamamlanma durumu:** Hangi modelin kaç fold'u bitirdiği.
- **Ham sayfalar:** Fold bazlı bütün metrikler.

Bazı gruplar henüz bitmemişse de çalıştırılabilir; o ana kadar tamamlanan fold'ları birleştirir ve eksikleri listeler.


In [ ]:
# ======================= TÜM SONUÇLARI BİRLEŞTİR =======================
# Dört grup notebook'u (1–4) tamamlandıktan sonra çalıştır. GPU gerekmez.
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_ROOT = Path("/content/drive/MyDrive/breast_mri_standard_results")   # grup notebook'larıyla aynı olmalı
CV_MODE = "holdout5"                                                       # grup notebook'larıyla aynı olmalı
GROUPS = ["baseline_cnn", "baseline_vit", "optimized_cnn", "optimized_vit"]
METRIC_COLS = ["Accuracy", "Precision", "Recall (Sensitivity)", "Specificity", "F1-Score", "ROC-AUC", "Loss"]

root = OUTPUT_ROOT / CV_MODE
results = [json.loads(p.read_text()) for p in sorted(root.glob("*/*/Fold_*/result.json"))]

# Hangi grup/model kaç fold tamamladı?
status = pd.DataFrame([{"Group": r["Group"], "Model": r["Model"], "Fold": r["Fold"]} for r in results])
progress = status.groupby(["Group", "Model"]).Fold.count().rename("Tamamlanan fold").reset_index()
display(progress)
eksik = progress[progress["Tamamlanan fold"] < 5]
if len(eksik): print("UYARI: 5 fold'u tamamlanmamış modeller var (yukarıda < 5 olanlar).")
for g in GROUPS:
    if g not in set(status.Group): print(f"UYARI: {g} grubunun hiç sonucu yok.")

rows = [{"Group": r["Group"], "Model": r["Model"], "Fold": r["Fold"], "Best Epoch": r["Best Epoch"],
         "Split": s, **m} for r in results for s, m in r["Splits"].items()]
per_fold = pd.DataFrame(rows)
g = per_fold.groupby(["Group", "Model", "Split"], sort=False)[METRIC_COLS]
summary = g.mean().add_suffix(" mean").join(g.std(ddof=1).add_suffix(" std")).reset_index()
article = per_fold.groupby(["Group", "Model", "Split"], sort=False)[METRIC_COLS[:-1]].agg(
    lambda s: f"{100*s.mean():.2f} ± {100*s.std(ddof=1):.2f}").reset_index()
order = {g: i for i, g in enumerate(GROUPS)}
article = article.sort_values(["Group", "Model"], key=lambda c: c.map(order) if c.name == "Group" else c)

fold_counts = root.parent / f"fold_counts_{CV_MODE}.csv"
with pd.ExcelWriter(root / "ALL_RESULTS.xlsx") as xw:
    for split in ["Test", "Validation", "Train"]:
        article[article.Split == split].to_excel(xw, sheet_name=f"{split} (ort±std)", index=False)
    if fold_counts.exists():
        pd.read_csv(fold_counts).to_excel(xw, sheet_name="Veri bölünmesi", index=False)
    progress.to_excel(xw, sheet_name="Tamamlanma durumu", index=False)
    summary.to_excel(xw, sheet_name="Ortalama-Std (ham)", index=False)
    per_fold.to_excel(xw, sheet_name="Fold bazlı (ham)", index=False)
print("Kaydedildi:", root / "ALL_RESULTS.xlsx")
display(article[article.Split == "Test"])
